# Zero offset correction: calibrate pressure sensor

## Load and inspect data
Load pickle file and inspect contents

In [ ]:
import numpy as np

# Import necessary pyologger utilities
from pyologger.utils.folder_manager import *
from pyologger.utils.event_manager import *
from pyologger.plot_data.plotter import *
from pyologger.calibrate_data.zoc import *
from pyologger.io_operations.base_exporter import *
from pyologger.analyze_data.find_segments import *

# dataset_id = "nesc-adult-hi-monk-seal_dive-imu_SR-MB"
# deployment_id = "2018-04-18_nesc-001"
# deployment_id = "2019-02-04_nesc-006"
# deployment_id = "2018-08-31_nesc-005"
# deployment_id = "2018-04-19_nesc-002"

dataset_id = "mile-adult-sese_vdr_argentina_RD-KM"
deployment_id = "2013-11-09_mile-008"


# dataset_id = "mian-juv-nese_po2_JM-PP"
# deployment_id = "2008-04-23_mian-022"

# dataset_id = "mile-adult-sese_vdr_argentina_RD-KM"
# deployment_id = "2012-11-01_mile-003"

#dataset_id = "caca-chmy_juv-adult_CW-PP-KS"
#deployment_id = "2015-09-20_caca-006" # "2015-09-19_caca-005" #"2015-09-06_caca-001"

# dataset_id = "apfo-adult-penguin_hr-sr-vid_penguin-ranch_JKB-PP"
# deployment_id = "2019-11-08_apfo-001"

# dataset_id = "oror-adult-orca_hr-sr-vid_sw_JKB-PP"
# deployment_id = "2023-10-26_oror-001"

# dataset_id = "mian-juv-nese_sleep_lml-ano_JKB"
# deployment_id = "2019-10-25_mian-001"
# deployment_id = "2021-04-17_mian-011"

# Load important file paths and configurations
config, data_dir, color_mapping_path, montage_path = load_configuration()
# Streamlit load data
animal_id, dataset_id, deployment_id, dataset_folder, deployment_folder, data_pkl, param_manager = select_and_load_deployment(
    data_dir, dataset_id=dataset_id, deployment_id=deployment_id
    )
pkl_path = os.path.join(deployment_folder, 'outputs', 'data.pkl')

In [ ]:
data_pkl.signal_info

In [ ]:
current_processing_step = "Processing Step 01 IN PROGRESS."
param_manager.add_to_config("current_processing_step", current_processing_step)

In [ ]:
critical_signal = 'pressure'

has_derived_depth = (
    hasattr(data_pkl, 'signal_data')
    and isinstance(data_pkl.signal_data, dict)
    and 'depth' in data_pkl.signal_data
    and data_pkl.signal_data['depth'] is not None
)

if has_derived_depth:
    print("✅ data_pkl.signal_data['depth'] already exists.")
    newdepth_df = data_pkl.signal_data['depth'].copy()
    if 'depth' in newdepth_df.columns and 'pressure' not in newdepth_df.columns:
        newdepth_df = newdepth_df.rename(columns={'depth': 'pressure'})
    if 'corrected_depth' in newdepth_df.columns and 'pressure' not in newdepth_df.columns:
        newdepth_df = newdepth_df.rename(columns={'corrected_depth': 'pressure'})
    data_pkl.signal_data['pressure'] = newdepth_df
    if hasattr(data_pkl, 'signal_info') and isinstance(data_pkl.signal_info, dict):
        depth_info = data_pkl.signal_info.get('depth')
        if depth_info is not None:
            if 'channels' in depth_info and depth_info['channels'] == ['depth']:
                depth_info['channels'] = ['pressure']
            if 'metadata' in depth_info and 'depth' in depth_info['metadata']:
                depth_info['metadata']['pressure'] = depth_info['metadata'].pop('depth')
            data_pkl.signal_info['depth'] = depth_info
            data_pkl.signal_info['pressure'] = depth_info
    print("✅ Copied depth data and metadata to pressure, with pressure column.")
    skip_step = False
else:
    # If critical signal doesn't exist, create flag to skip step.
    if critical_signal not in data_pkl.signal_data or data_pkl.signal_data[critical_signal] is None:
        print(f"⚠️ Signal: {critical_signal} not found. Skipping processing.")
        skip_step = True
        print(f'‼️ DO NOT PROCEED - Skip_step: {skip_step} due to missing critical signal: {critical_signal}')
    elif 'pressure' not in data_pkl.signal_data[critical_signal].columns:
        print("⚠️ Pressure column not found in pressure signal. Skipping processing.")
        skip_step = True
    else:
        # signals exists and can be processed normally
        skip_step = False
        print(f'✅ Proceed - Skip_step: {skip_step}. Critical signal: {critical_signal} found.')

In [ ]:
skip_step

## Find dives
Involves a zero offset correction with `zoc`

In [ ]:
data_pkl.signal_data['pressure']

In [ ]:
data_pkl.signal_info['pressure']['original_units']

In [ ]:
# Check that units are in meters or convert if necessary
original_pressure_unit = data_pkl.signal_info['pressure']['original_units']
pressure_unit = data_pkl.signal_info['pressure']['units']

if original_pressure_unit == 'bar' and pressure_unit != 'm': # if bar to m and hasn't been converted yet
    print("Converting pressure from bar to m")
    data_pkl.signal_data['pressure']['pressure'] *= 10
    data_pkl.signal_info['pressure']['units'] = 'm'
    print("✅ Pressure unit changed from bar to m")
elif original_pressure_unit == 'cm':
    print("Converting pressure from cm to m")
    data_pkl.signal_data['pressure']['pressure'] /= 100
    data_pkl.signal_info['pressure']['units'] = 'm'
    print("✅ Pressure unit changed from cm to m")
elif original_pressure_unit in ['m', '100bar', '100bar_1', '30bar_1', 'msw']: # including CATS format weird 100bar_1 which seems to be m
    print("✅ Pressure unit already in m")
    data_pkl.signal_info['pressure']['units'] = 'm'
    pass
else:
    print(f"Unknown pressure unit: {pressure_unit}")
    raise ValueError(f"Unknown pressure unit: {pressure_unit}")

new_pressure_unit = data_pkl.signal_info['pressure']['units']




In [ ]:
restart_threshold_setting = param_manager.get_from_config(
    variable_names=["logger_restart_pressure_threshold"],
    section="dive_detection_settings"
).get("logger_restart_pressure_threshold")
try:
    logger_restart_pressure_threshold = float(restart_threshold_setting)
except (TypeError, ValueError):
    logger_restart_pressure_threshold = -500.0

# 1. Check if logger is known to produce extreme pressure values
if data_pkl.signal_info['pressure']['logger_manufacturer'] == 'Evolocus':
    # 2. Check if logger_restart events have already been added
    if data_pkl.event_data is None or data_pkl.event_data.empty or not any(data_pkl.event_data['key'] == 'logger_restart'):
        # 3. Identify bad segments based on unrealistic negative pressure
        pressure_df = data_pkl.signal_data['pressure'].copy()
        restarts = find_segments(
            data=pressure_df,
            column='pressure',
            criteria=lambda x: x < logger_restart_pressure_threshold,
            min_duration=None
        )

        # 4. Add restart events using standardized event creation
        if not restarts.empty:
            data_pkl.event_data = create_state_event(
                state_df=restarts,
                key="logger_restart",
                start_time_column="start_datetime",
                duration_column="duration",
                description="Detected logger restart from extreme pressure",
                long_description=(
                "Logger restart inferred from pressure values dropping below "
                f"{logger_restart_pressure_threshold}, typically inserted by logger hardware during reboot."
            ),
                existing_events=data_pkl.event_data
            )
            print(f"🟠 Added {len(restarts)} logger_restart event(s) to event_data.")

            # 5. Replace pressure values with NaN around each segment
            pressure_series = data_pkl.signal_data['pressure']
            datetimes = pressure_series['datetime']
            for _, row in restarts.iterrows():
                start = row['start_datetime']
                end = row['end_datetime']
                mask = (datetimes >= start) & (datetimes <= end)
                buffer_before = datetimes.shift(1)
                buffer_after = datetimes.shift(-1)
                buffer_mask = (buffer_before >= start) & (buffer_before <= end) | (buffer_after >= start) & (buffer_after <= end)
                full_mask = mask | buffer_mask
                data_pkl.signal_data['pressure'].loc[full_mask, 'pressure'] = np.nan
            print(
                "⚠️ Replaced extreme pressure values "
                f"(< {logger_restart_pressure_threshold}) and surrounding buffer with NaN."
            )
        else:
            print("✅ No restart segments detected.")
    else:
        print("✅ Logger restart events already exist in event_data.")

    # 6. Check again in case any extreme values remain outside known segments
    extreme_exists = (data_pkl.signal_data['pressure']['pressure'] < logger_restart_pressure_threshold).any()
    if extreme_exists:
        data_pkl.signal_data['pressure'].loc[
            data_pkl.signal_data['pressure']['pressure'] < logger_restart_pressure_threshold, 'pressure'
        ] = np.nan
        print(
            "⚠️ Replaced residual extreme pressure values "
            f"(< {logger_restart_pressure_threshold}) with NaN."
        )
    else:
        print("✅ No extreme pressure values found.")
else:
    print("✅ No logger restart check needed for this logger.")

In [ ]:
data_pkl.signal_data['pressure']['pressure'].min()

In [ ]:
# Step 4: Load depth and temperature data
depth_data = data_pkl.signal_data["pressure"]["pressure"]
depth_datetime = data_pkl.signal_data["pressure"]["datetime"]
depth_fs = data_pkl.signal_info["pressure"]["sampling_frequency"]
if 'temperature-ext' in data_pkl.signal_data:
    temp_data = data_pkl.signal_data['temperature-ext']['temp-ext']
    temp_fs = data_pkl.signal_info['temperature-ext']['sampling_frequency']
elif 'temperature-int' in data_pkl.signal_data:
    temp_data = data_pkl.signal_data['temperature-int']['temp-int']
    temp_fs = data_pkl.signal_info['temperature-int']['sampling_frequency']
else:
    temp_data = None
    temp_fs = None

In [ ]:
data_pkl.signal_data['pressure']


In [ ]:
data_pkl.event_data

In [ ]:
len(np.unique(depth_data))

In [ ]:
# **Step 2: Load Configuration Parameters**
dive_detection_settings = param_manager.get_from_config(
    variable_names=[
        "first_deriv_threshold", "min_duration", "depth_threshold",
        "apply_temp_correction", "min_depth_threshold", "dive_duration_threshold",
        "smoothing_window", "downsampled_sampling_rate", "baseline_adjust",
        "use_flat_chunks", "min_flat_chunks_for_zoc",
        "disable_automatic_sign_flipping", "conversion_factor",
        "logger_restart_pressure_threshold"
    ],
    section="dive_detection_settings"
)

# Default settings
default_settings = {
    "first_deriv_threshold": 0.1,       # Threshold for the first derivative of depth (meters/second)
    "min_duration": 10,                 # Minimum duration for a surface interval (seconds)
    "depth_threshold": 5,               # Maximum depth to qualify as a surface interval (meters)
    "apply_temp_correction": False,     # Apply temperature correction during zero offset correction (True/False)
    "min_depth_threshold": 0.5,         # Minimum depth to start/end a dive (meters)
    "dive_duration_threshold": 10,      # Minimum duration for a dive (seconds)
    "smoothing_window": 5,              # Smoothing window
    "downsampled_sampling_rate": 1,     # Target sampling rate after downsampling (default 1Hz)
    "baseline_adjust": 0,               # Adjust baseline depth data if needs to be raised or lowered
    "use_flat_chunks": True,            # Use flat-chunk-based ZOC
    "min_flat_chunks_for_zoc": 10,      # Minimum flat chunks required to apply ZOC
    "disable_automatic_sign_flipping": False,  # If True, skip automatic sign inversion
    "conversion_factor": 1.0,           # Additional scaling factor for pressure/depth (0-100)
    "logger_restart_pressure_threshold": -500.0  # Restart sentinel threshold (values below are removed)
}

# If settings are missing or None, initialize them
if dive_detection_settings is None:
    dive_detection_settings = default_settings.copy()
else:
    # Fill missing/None keys from defaults
    dive_detection_settings = {
        k: dive_detection_settings.get(k) if dive_detection_settings.get(k) is not None else v
        for k, v in default_settings.items()
    }

# Save if changes were made
param_manager.add_to_config(entries=dive_detection_settings, section="dive_detection_settings")

# Step 6: Process depth data - Downsample, smooth, adjust baseline, and calculate first derivative
# Interpolate NaNs for processing
interpolated_depth_data = depth_data.interpolate(limit_direction='both')

depth_processing_params = {
    "original_sampling_rate": depth_fs,
    "downsampled_sampling_rate": int(dive_detection_settings["downsampled_sampling_rate"]),
    "baseline_adjust": dive_detection_settings["baseline_adjust"]
}



In [ ]:
conversion_factor = float(dive_detection_settings.get("conversion_factor", 1.0))
conversion_factor = max(0.0, min(100.0, conversion_factor))
dive_detection_settings["conversion_factor"] = conversion_factor
if conversion_factor != 1.0:
    data_pkl.signal_data['pressure'].loc[:, 'pressure'] *= conversion_factor
    depth_data = depth_data * conversion_factor
    interpolated_depth_data = interpolated_depth_data * conversion_factor
    print(f"✅ Applied conversion_factor={conversion_factor}")
else:
    print("✅ conversion_factor is 1.0; no additional pressure scaling applied.")

first_derivative, downsampled_depth = smooth_downsample_derivative(interpolated_depth_data, **depth_processing_params)

# Ensure pressure/depth values are mostly positive after manual baseline adjustment.
# Run this check on the baseline-adjusted downsampled signal so sign decision reflects user baseline settings.
disable_automatic_sign_flipping = bool(dive_detection_settings.get("disable_automatic_sign_flipping", False))
downsampled_valid = pd.Series(downsampled_depth).dropna()
if disable_automatic_sign_flipping:
    print("ℹ️ Automatic sign flipping is disabled by config; skipping sign check.")
elif downsampled_valid.empty:
    print("⚠️ No valid baseline-adjusted depth values to evaluate sign (all NaN).")
else:
    neg_count = (downsampled_valid < 0).sum()
    pos_count = (downsampled_valid > 0).sum()
    zero_count = (downsampled_valid == 0).sum()
    print(f"📊 Post-baseline sign check — negative: {neg_count}, positive: {pos_count}, zero: {zero_count}")

    if neg_count > pos_count:
        # Keep all depth representations consistent if sign flip is needed.
        data_pkl.signal_data['pressure'].loc[:, 'pressure'] *= -1
        depth_data *= -1
        interpolated_depth_data *= -1
        downsampled_depth *= -1
        first_derivative *= -1
        print("🔄 Baseline-adjusted depth was mostly negative; multiplied by -1 to make it positive.")
    else:
        print("✅ Baseline-adjusted depth is not mostly negative; no sign change applied.")
# Adjust datetime indexing based on the new downsample rate
depth_downsampled_datetime = depth_datetime.iloc[::int(depth_fs / dive_detection_settings["downsampled_sampling_rate"])]



In [ ]:
# Ensure indexing does not go out of bounds
if len(depth_downsampled_datetime) > len(downsampled_depth):
    depth_downsampled_datetime = depth_downsampled_datetime[:len(downsampled_depth)]

# Print summary of processing
print(f"✅ Depth processing complete: Downsampled to {dive_detection_settings['downsampled_sampling_rate']} Hz")
print(f"✅ Baseline adjustment applied: {dive_detection_settings['baseline_adjust']} meters")

# Detect flat chunks (potential surface intervals)
flat_chunk_params = {
    "depth": downsampled_depth,
    "datetime_data": depth_downsampled_datetime,
    "first_derivative": first_derivative,
    "threshold": dive_detection_settings["first_deriv_threshold"],
    "min_duration": dive_detection_settings["min_duration"],
    "depth_threshold": dive_detection_settings["depth_threshold"],
    "original_sampling_rate": depth_fs,
    "downsampled_sampling_rate": dive_detection_settings["downsampled_sampling_rate"]
}
flat_chunks = detect_flat_chunks(**flat_chunk_params)
flat_chunks

In [ ]:
use_flat_chunks = bool(dive_detection_settings.get("use_flat_chunks", True))
min_flat_chunks_for_zoc = int(dive_detection_settings.get("min_flat_chunks_for_zoc", 10))
enough_flat_chunks = len(flat_chunks) >= min_flat_chunks_for_zoc

if use_flat_chunks and enough_flat_chunks:
    # Apply zero offset correction
    zoc_params = {
        "depth": downsampled_depth,
        "temp": temp_data.values if temp_data is not None else None,
        "flat_chunks": flat_chunks
    }
    corrected_depth_temp, corrected_depth_no_temp, depth_correction = apply_zero_offset_correction(**zoc_params)
    pressure = corrected_depth_temp if dive_detection_settings["apply_temp_correction"] else corrected_depth_no_temp
    print(f"✅ Applied ZOC using {len(flat_chunks)} flat chunks (minimum required: {min_flat_chunks_for_zoc}).")
else:
    # Fallback: use baseline-adjusted depth when ZOC is disabled or too few flat chunks are available
    pressure = downsampled_depth.copy()
    depth_correction = np.zeros_like(downsampled_depth)
    if not use_flat_chunks:
        print("ℹ️ Skipped ZOC because use_flat_chunks is False. Using baseline-adjusted depth only.")
    else:
        print(
            f"ℹ️ Skipped ZOC due to insufficient flat chunks: {len(flat_chunks)} found, "
            f"{min_flat_chunks_for_zoc} required. Using baseline-adjusted depth only."
        )

# Detect dives using find_segments
depth_df = pd.DataFrame({
    'datetime': depth_downsampled_datetime,
    'depth': pressure
})


In [ ]:
depth_df

In [ ]:
len(np.unique(pressure))

In [ ]:
dives = find_segments(
    data=depth_df,
    column='depth',
    criteria=lambda x: x > dive_detection_settings['min_depth_threshold'],
    min_duration=dive_detection_settings['dive_duration_threshold'],
)

nan_mask = depth_data.isna().reindex(depth_downsampled_datetime.index, method='nearest')
dives['has_nans'] = dives.apply(lambda row: nan_mask.loc[(depth_downsampled_datetime >= row['start_datetime']) & (depth_downsampled_datetime <= row['end_datetime'])].any(), axis=1)
dives['short_description'] = dives['has_nans'].apply(lambda x: 'dive-with-nan_start' if x else 'dive_start')

pressure = enforce_surface_before_after_dives(pressure, depth_downsampled_datetime, dives)

dives['dive_duration'] = (dives['end_datetime'] - dives['start_datetime']).dt.total_seconds()

transformation_log = [
    f"downsampled_{dive_detection_settings['downsampled_sampling_rate']}Hz",
    f"smoothed_{dive_detection_settings['smoothing_window']}s",
    f"conversion_factor_{dive_detection_settings['conversion_factor']}",
    f"ZOC_settings__first_deriv_threshold_{dive_detection_settings['first_deriv_threshold']}mps__min_duration_{dive_detection_settings['min_duration']}s__depth_threshold_{dive_detection_settings['depth_threshold']}m",
    f"DIVE_detection_settings__min_depth_threshold_{dive_detection_settings['min_depth_threshold']}m__dive_duration_threshold_{dive_detection_settings['dive_duration_threshold']}s__smoothing_window_{dive_detection_settings['smoothing_window']}"
]

print(f"✅ {len(flat_chunks)} surface intervals detected.")
print(f"✅ {len(dives)} dives detected.")
print("📖 Transformation Log:", transformation_log)

### Save data

In [ ]:
dives

In [ ]:
from pyologger.analyze_data.analyze_segments import *
# Append max depth for each dive segment
dives = append_stats(
    data=depth_df, 
    segment_df=dives, 
    statistics=[("max", "depth")]
)

In [ ]:
dives

In [ ]:
# Generate and update dive events
data_pkl.event_data = create_state_event(
    state_df=dives,
    key='dive',
    value_column='depth_max',
    start_time_column='start_datetime',
    duration_column='dive_duration', # in seconds
    description='dive_start',
    existing_events=data_pkl.event_data  # Pass existing events for overwrite and concatenation
)

# Update event_info with unique keys
data_pkl.event_info = list(data_pkl.event_data['key'].unique())

In [ ]:
data_pkl.event_info

## Save data to pickle

In [ ]:
# Create the derived_from_signals list
derived_from_signals = ["pressure"]
original_name = 'Temp-corrected Depth (m)' if dive_detection_settings["apply_temp_correction"] else 'Corrected Depth (m)'

# Save the corrected depth back to the data structure
depth_df = pd.DataFrame({
    'datetime': depth_downsampled_datetime,
    'depth': pressure
})
derived_info = {
    "channels": ["depth"],
    "metadata": {
            'depth': {'original_name': original_name,
                    'unit': 'm',
                    'signal': 'pressure'}
    },
    "derived_from_signals": derived_from_signals.append("temperature") if dive_detection_settings["apply_temp_correction"] else derived_from_signals,
    "transformation_log": transformation_log.append("temperature_correction") if dive_detection_settings["apply_temp_correction"] else derived_from_signals
}

data_pkl.signal_data['depth'] = depth_df
data_pkl.derived_info['depth'] = derived_info

In [ ]:
# Retrieve necessary time settings from the settings section
time_settings = param_manager.get_from_config(
    ["overlap_start_time", "overlap_end_time", "zoom_window_start_time", "zoom_window_end_time"],
    section="settings"
)

# Assign retrieved values to variables
OVERLAP_START_TIME = time_settings.get("overlap_start_time")
OVERLAP_END_TIME = time_settings.get("overlap_end_time")
ZOOM_START_TIME = time_settings.get("zoom_window_start_time")
ZOOM_END_TIME = time_settings.get("zoom_window_end_time")

# Confirm the values or raise an error if any are missing
if None in {OVERLAP_START_TIME, OVERLAP_END_TIME, ZOOM_START_TIME, ZOOM_END_TIME}:
    raise ValueError("One or more required time values were not found in the config file.")

# Display the loaded values
print("OVERLAP_START_TIME:", OVERLAP_START_TIME)
print("OVERLAP_END_TIME:", OVERLAP_END_TIME)
print("ZOOM_START_TIME:", ZOOM_START_TIME)
print("ZOOM_END_TIME:", ZOOM_END_TIME)

In [ ]:
fig = plot_tag_data_interactive(
    data_pkl=data_pkl,
    signals=['pressure','depth'],
    time_range=(depth_downsampled_datetime.min(), depth_downsampled_datetime.max()),
    note_annotations={"dive": {"signal": "depth", "symbol": "triangle-down", "color": "blue"}},
    state_annotations={"dive": {"signal": "depth", "color": "rgba(150, 150, 150, 0.3)"}},
    color_mapping_path=color_mapping_path,
    target_sampling_rate=1
)
fig.show_dash(mode="inline")

In [ ]:
from pyologger.io_operations.base_exporter import *

exporter = BaseExporter(data_pkl) # Create a BaseExporter instance using data pickle object
netcdf_file_path = os.path.join(deployment_folder, 'outputs', f'{deployment_id}_step01.nc') # Define the export path
exporter.save_to_netcdf(data_pkl, filepath=netcdf_file_path) # Save to NetCDF format

In [ ]:
import xarray as xr

# Define the path to the NetCDF file
netcdf_path = os.path.join(deployment_folder, 'outputs', f'{deployment_id}_step01.nc')

# Open the NetCDF file
data = xr.open_dataset(netcdf_path)

# Display the contents of the NetCDF file
display(data)

In [ ]:
# Dynamically generate and print the statement
print(f"`{', '.join(derived_from_signals)}` signal data was transformed into derived data `depth` by applying these transformations: {', '.join(transformation_log)}")
current_processing_step = "Processing Step 01. Calibration of pressure signal complete and dives analyzed."
print(current_processing_step)

In [ ]:
# Add or update the current_processing_step for the specified deployment
print(current_processing_step)
param_manager.add_to_config("current_processing_step", current_processing_step)

# Optional: save new pickle file
with open(pkl_path, 'wb') as file:
        pickle.dump(data_pkl, file)
print("Pickle file updated.")